# Idiom-Aware Fine-Tuning of EN-DE Translations

**Idea**: 2-stage fine-tuning of a pre-trained model to increase idiom translation accuracy without sacrificing general language translation quality

---
**Stages**

---



**Stage 0**: Baseline (Helsikin-NLP model)

**Stage 1**: Immediate Idiom-only fine-tuning
- Increase the model's idiom translation performance by fine-tuning the baseline from stage 0 on EN-DE idiom sentence pairs (`IdiomsInCtx-MT`)
- This will likely increase idiom-specific translation performance (measured as `Idiom BLEU`), and likely decrease general language translation (measured as `WMT BLEU)` performance.

**Stage 2**: General Language fine-tuning
- Fine-tune the model from stage 1 on a general language EN-DE dataset (`WMT-14`)
- The hope is to maintain the learned idiom-translation ability from stage 1, while recovering some of the lost general language translation perfromance.

Ideally, our stage 2 model has not sacrificied its general language translation capabilties, but has notably increased idiom-specific translation capabilities.

---
**Approach in this notebook**

---

- Mount Drive
- Install Libraries and Dependencies
- Load Idiom and WMT data
- Create Metrics Logger and write results to results/metrics.csv
- Load Baseline Model
- Evaluate Baseline Model
- Fine-tune Stage 1
- Evaluate Stage 1
- Fine-tune Stage 2
- Evaluate Stage 2


## Clone Repo from Git / Commit Changes
Run the git clone everytime a new session is started

In [1]:
!rm -rf Idiom-Aware-Finetuning-in-MT-EN-to-DE
!git clone https://github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
%cd Idiom-Aware-Finetuning-in-MT-EN-to-DE
!ls

def clear_generation_max_length(model):
    if hasattr(model, "generation_config") and model.generation_config is not None:
        try:
            model.generation_config.max_length = None
        except Exception:
            pass
    if hasattr(model, "config") and model.config is not None:
        try:
            model.config.max_length = None
        except Exception:
            pass
    return model


# Reproducibility / rerun controls
FORCE_RETRAIN = False

def checkpoint_exists(model_dir):
    if not os.path.isdir(model_dir):
        return False
    has_config = os.path.exists(os.path.join(model_dir, "config.json"))
    has_weights = (
        os.path.exists(os.path.join(model_dir, "pytorch_model.bin")) or
        os.path.exists(os.path.join(model_dir, "model.safetensors"))
    )
    return has_config and has_weights


Cloning into 'Idiom-Aware-Finetuning-in-MT-EN-to-DE'...
remote: Enumerating objects: 86, done.
remote: Counting objects: 100% (86/86), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 86 (delta 36), reused 34 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (86/86), 215.01 KiB | 3.91 MiB/s, done.
Resolving deltas: 100% (36/36), done.
/content/Idiom-Aware-Finetuning-in-MT-EN-to-DE
configs  notebooks  README.md  results	scripts  src


Copy notebook from Drive into Repo

In [2]:
# !cp "/content/drive/MyDrive/ds266_idiom_mt/DS266_Idiom_Aware_MT_ALL_IN_ONE.ipynb" notebooks/

Add and commit changes

In [3]:
#!git config --global user.email "auntiepersilla@gmail.com"
#!git config --global user.name "Michael Strommer"

#!git add notebooks/DS266_Idiom_Aware_MT_ALL_IN_ONE.ipynb
#!git commit -m "Add all-in-one Drive-first pipeline notebook"

Push to GitHub

In [4]:
#!git remote set-url origin https://auntiepersilla:<TOKEN>@github.com/auntiepersilla/Idiom-Aware-Finetuning-in-MT-EN-to-DE.git
#!git push origin main

## Mount Drive, Create Paths

In [5]:
from google.colab import drive
drive.mount("/content/drive")

import os
DRIVE_ROOT = "/content/drive/MyDrive/ds266_idiom_mt"
CHECKPOINT_DIR = os.path.join(DRIVE_ROOT, "checkpoints")
RESULTS_DIR = os.path.join(DRIVE_ROOT, "results")
CACHE_DIR = os.path.join(DRIVE_ROOT, "cache")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = CACHE_DIR
os.environ["HF_DATASETS_CACHE"] = CACHE_DIR

print("DRIVE_ROOT:", DRIVE_ROOT)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("CACHE_DIR:", CACHE_DIR)

Mounted at /content/drive
DRIVE_ROOT: /content/drive/MyDrive/ds266_idiom_mt
CHECKPOINT_DIR: /content/drive/MyDrive/ds266_idiom_mt/checkpoints
RESULTS_DIR: /content/drive/MyDrive/ds266_idiom_mt/results
CACHE_DIR: /content/drive/MyDrive/ds266_idiom_mt/cache


## Install Dependencies & Imports

In [6]:
!pip -q install -U "transformers>=4.38" datasets evaluate sacrebleu accelerate sentencepiece sacremoses "protobuf<6"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 74.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 57.3 MB/s eta 0:00:00


In [7]:
import random, numpy as np, torch
from datasets import load_dataset, DatasetDict
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments
import sacrebleu
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


## Load Data

- Idiom-only Data: davidstap/IdiomsInCtx-MT (1000 EN-DE idiom sentence pairs)
  - train: 800
  - dev: 100
  - test: 100
- General Translation Training and Test Data: WMT-14 EN-DE
  - train: 20.000
  - test: 2.000
  - test: 3.003

In [8]:
def load_idioms(name="davidstap/IdiomsInCtx-MT", config="en-de", train_frac=0.8, dev_frac=0.1, seed=42):
    raw = load_dataset(name, config)

    full = raw["test"] if ("test" in raw and len(raw.keys()) == 1) else raw[list(raw.keys())[0]]
    src_lang, tgt_lang = config.split("-")

    def standardize(ex):
        if "translation" in ex and src_lang in ex["translation"] and tgt_lang in ex["translation"]:
            return {"src": ex["translation"][src_lang], "tgt": ex["translation"][tgt_lang]}
        if src_lang in ex and tgt_lang in ex:
            return {"src": ex[src_lang], "tgt": ex[tgt_lang]}
        raise ValueError(list(ex.keys()))

    full = full.map(standardize)
    full = full.remove_columns([c for c in full.column_names if c not in ["src","tgt"]])

    tmp = full.train_test_split(test_size=(1-train_frac), seed=seed)
    train, rest = tmp["train"], tmp["test"]

    dev_frac_of_rest = dev_frac / (1-train_frac)
    tmp2 = rest.train_test_split(test_size=(1-dev_frac_of_rest), seed=seed)
    dev, test = tmp2["train"], tmp2["test"]

    return DatasetDict({"train": train, "dev": dev, "test": test})

def load_wmt14(ft_train_n=20000, ft_dev_n=2000, seed=42):
    wmt = load_dataset("wmt14", "de-en")
    def to_en_de(ex):
        tr = ex["translation"]
        return {"src": tr["en"], "tgt": tr["de"]}

    train = wmt["train"].map(to_en_de, remove_columns=wmt["train"].column_names)
    test = wmt["test"].map(to_en_de, remove_columns=wmt["test"].column_names)

    shuf = train.shuffle(seed=seed)
    ft_train = shuf.select(range(min(ft_train_n, len(shuf))))
    ft_dev = shuf.select(range(min(ft_train_n, len(shuf)), min(ft_train_n+ft_dev_n, len(shuf))))
    return DatasetDict({"ft_train": ft_train, "ft_dev": ft_dev, "wmt_test": test})

idiom_ds = load_idioms(seed=SEED)
general_ds = load_wmt14(seed=SEED)

print("idiom_ds:", {k: len(v) for k,v in idiom_ds.items()})
print("general_ds:", {k: len(v) for k,v in general_ds.items()})
print("idiom sample:", idiom_ds["test"][0])

idiom_ds: {'train': 800, 'dev': 100, 'test': 100}
general_ds: {'ft_train': 20000, 'ft_dev': 2000, 'wmt_test': 3003}
idiom sample: {'src': 'Not a word of all this to anyone!', 'tgt': 'Kein Wort von all dem zu irgendjemandem!'}


## Metrics and Generation Helpers

In [9]:
# Set Global Hyperparameters for Predictions
max_source_len = 256
max_new_tokens = 128
batch_size = 16

In [10]:
@torch.no_grad()

# Generate Predictions
def generate_preds(model, tok, sources, max_source_len=max_source_len, max_new_tokens=max_new_tokens, batch_size=batch_size):
    model.eval()
    preds = []

    # Avoid the repeated Transformers warning about both max_length and max_new_tokens
    # by explicitly clearing max_length on a copied generation config.
    import copy
    gen_cfg = copy.deepcopy(model.generation_config)
    gen_cfg.max_length = None

    for i in range(0, len(sources), batch_size):
        batch = sources[i:i+batch_size]
        inputs = tok(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=max_source_len,
        ).to(device)
        out = model.generate(
            **inputs,
            generation_config=gen_cfg,
            max_new_tokens=max_new_tokens,
        )
        preds.extend(tok.batch_decode(out, skip_special_tokens=True))
    return preds

# Compute Metrics
def compute_metrics(preds, refs):
    bleu = sacrebleu.corpus_bleu(preds, [refs]).score
    chrf = sacrebleu.corpus_chrf(preds, [refs]).score
    return {"bleu": float(bleu), "chrf": float(chrf)}

In [11]:
# Metrics Logger

METRICS_PATH = os.path.join(RESULTS_DIR, "metrics.csv")

METRIC_FIELDS = [
    "timestamp",
    "run_name",
    "split",
    "bleu",
    "chrf",
    "model_name",
    "learning_rate",
    "epochs",
    "batch_size",
    "max_source_len",
    "max_new_tokens",
    "freeze_encoder",
    "train_size",
    "dev_size",
    "seed",
    "n_eval",
    "notes",
]



# Canonical automatic-metrics file used downstream for clean analysis merges
CANONICAL_METRICS_PATH = os.path.join(RESULTS_DIR, "automatic_metrics_canonical.csv")

CANONICAL_FIELDS = [
    "timestamp",
    "model",
    "model_label",
    "split",
    "bleu",
    "chrf",
    "source_notebook",
    "source_run_name",
    "notes",
]



def infer_model_label(model, source_run_name=""):
    model_str = str(model)
    run_str = str(source_run_name)
    if model_str == "baseline" or run_str == "baseline":
        return "baseline"
    if model_str == "idiom_only_v1" or run_str == "idiom_only_v1":
        return "idiom_only_v1"
    if model_str == "two_stage_frozen_v1" or "two_stage_frozen" in model_str or "two_stage_frozen" in run_str:
        return "two_stage_frozen_v1"
    return model_str

def upsert_canonical_metric(model, split, metrics, *, source_notebook, source_run_name, notes=""):
    row = {
        "timestamp": datetime.now(ZoneInfo('America/Los_Angeles')).isoformat(timespec="seconds"),
        "model": model,
        "model_label": infer_model_label(model, source_run_name),
        "split": split,
        "bleu": float(metrics.get("bleu", float("nan"))),
        "chrf": float(metrics.get("chrf", float("nan"))),
        "source_notebook": source_notebook,
        "source_run_name": source_run_name,
        "notes": notes,
    }

    df_new = pd.DataFrame([row], columns=CANONICAL_FIELDS)

    if os.path.exists(CANONICAL_METRICS_PATH):
        df_old = pd.read_csv(CANONICAL_METRICS_PATH)
        df = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df = df_new

    # Keep timestamps as ISO strings and sort lexicographically.
    # This avoids timezone/parsing inconsistencies while preserving chronological order.
    df["timestamp"] = df["timestamp"].astype(str)
    df = df.sort_values(["model", "split", "timestamp"])
    df = df.drop_duplicates(subset=["model_label", "split"], keep="last")
    df.to_csv(CANONICAL_METRICS_PATH, index=False)
    return df_new

def log_metrics(
    run_name,
    split,
    metrics,
    *,
    model_name=None,
    learning_rate=None,
    epochs=None,
    batch_size=None,
    max_source_len=None,
    max_new_tokens=None,
    freeze_encoder=None,
    train_size=None,
    dev_size=None,
    seed=None,
    n_eval=None,
    notes=None,
):
    # Initialize full schema with None
    row = {k: None for k in METRIC_FIELDS}

    row.update({
        "timestamp": datetime.now(ZoneInfo('America/Los_Angeles')).isoformat(timespec="seconds"),
        "run_name": run_name,
        "split": split,
        "bleu": float(metrics.get("bleu", float("nan"))),
        "chrf": float(metrics.get("chrf", float("nan"))),
        "model_name": model_name,
        "learning_rate": learning_rate,
        "epochs": epochs,
        "batch_size": batch_size,
        "max_source_len": max_source_len,
        "max_new_tokens": max_new_tokens,
        "freeze_encoder": freeze_encoder,
        "train_size": train_size,
        "dev_size": dev_size,
        "seed": seed,
        "n_eval": n_eval,
        "notes": notes,
    })

    df = pd.DataFrame([row])

    if os.path.exists(METRICS_PATH):
        df.to_csv(METRICS_PATH, mode="a", header=False, index=False)
    else:
        df.to_csv(METRICS_PATH, index=False)

    return row

### Canonical automatic-metrics writes

This notebook still appends to `results/metrics.csv` as a historical log, but it also upserts clean canonical rows to `results/automatic_metrics_canonical.csv`. For notebook 01, the canonical models owned by this notebook are:

- `baseline`
- `idiom_only_v1`
- `two_stage_frozen_v1`

The canonical file is keyed by `model_label` and `split`, so reruns update the latest row instead of creating duplicates.

### Checkpoint reuse behavior

This notebook now checks whether the expected checkpoint directories already exist before retraining. If a valid checkpoint is present and `FORCE_RETRAIN = False`, the notebook will reuse the saved model instead of retraining it. Set `FORCE_RETRAIN = True` only when you intentionally want to overwrite or refresh the checkpoint.

# Stage 0: Baseline
- Model: Helsinki-NLP/opus-mt-en-de

In [12]:
# Load Model and Tokenizer
BASE_MODEL = "Helsinki-NLP/opus-mt-en-de"
tok = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False)
model = clear_generation_max_length(AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL).to(device))

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [13]:
#model.config

In [14]:
# Set test data
idiom_test = idiom_ds["test"]
wmt_test = general_ds["wmt_test"]

# Global Prediction Hyperparameters
MAX_SOURCE_LEN = 256
MAX_NEW_TOKENS = 128
BATCH_SIZE = 16

### Baseline Evaluation

In [15]:
# Evaluate Idiom Translation
idiom_pred = generate_preds(model, tok, idiom_test["src"],
                            max_source_len=MAX_SOURCE_LEN,
                            max_new_tokens=MAX_NEW_TOKENS,
                            batch_size=BATCH_SIZE)
idiom_m = compute_metrics(idiom_pred, idiom_test["tgt"])
print("baseline idioms:", idiom_m)

log_metrics(
    "baseline", "idioms_test", idiom_m,
    model_name=BASE_MODEL,
    learning_rate=None,
    epochs=None,
    batch_size=BATCH_SIZE,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(idiom_ds["train"]),
    dev_size=len(idiom_ds["dev"]),
    seed=SEED,
    n_eval=len(idiom_test),
    notes="pretrained baseline",
)

upsert_canonical_metric(
    "baseline",
    "idioms_test",
    idiom_m,
    source_notebook="01_experiment_idiom_aware_MT_two_stage_frozen_encoder.ipynb",
    source_run_name="baseline",
    notes="pretrained baseline",
)

# Evaluate General Translation
wmt_pred = generate_preds(model, tok, wmt_test["src"],
                          max_source_len=MAX_SOURCE_LEN,
                          max_new_tokens=MAX_NEW_TOKENS,
                          batch_size=BATCH_SIZE)
wmt_m = compute_metrics(wmt_pred, wmt_test["tgt"])
print("baseline wmt:", wmt_m)

log_metrics(
    "baseline", "wmt_test", wmt_m,
    model_name=BASE_MODEL,
    learning_rate=None,
    epochs=None,
    batch_size=BATCH_SIZE,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(general_ds["ft_train"]),
    dev_size=len(general_ds["ft_dev"]),
    seed=SEED,
    n_eval=len(wmt_test),
    notes="pretrained baseline (WMT test full 3003)",
)

upsert_canonical_metric(
    "baseline",
    "wmt_test",
    wmt_m,
    source_notebook="01_experiment_idiom_aware_MT_two_stage_frozen_encoder.ipynb",
    source_run_name="baseline",
    notes="pretrained baseline (WMT test full 3003)",
)


Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


baseline idioms: {'bleu': 39.64935194574556, 'chrf': 60.786111999956105}
baseline wmt: {'bleu': 27.567994028871393, 'chrf': 58.43184557779851}


,timestamp,model,model_label,split,bleu,chrf,source_notebook,source_run_name,notes
0,2026-03-28T21:13:27-07:00,baseline,baseline,wmt_test,27.567994,58.431846,01_experiment_idiom_aware_MT_two_stage_frozen_...,baseline,pretrained baseline (WMT test full 3003)


## Stage 1: Immediate idiom-focused fine-tuning
Training Function (with Encoder Freezing option)

In [16]:
def fine_tune(model_name_or_path, train_ds, dev_ds, out_dir, lr, epochs, batch_size=8, freeze_encoder=False):
    tok = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=False)
    model = clear_generation_max_length(AutoModelForSeq2SeqLM.from_pretrained(model_name_or_path).to(device))

    if freeze_encoder and hasattr(model, "model") and hasattr(model.model, "encoder"):
        for p in model.model.encoder.parameters():
            p.requires_grad = False
        if hasattr(model.model, "shared"):
            for p in model.model.shared.parameters():
                p.requires_grad = False

    def tokenize(batch):
        x = tok(batch["src"], truncation=True, max_length=256)
        y = tok(text_target=batch["tgt"], truncation=True, max_length=256)
        x["labels"] = y["input_ids"]
        return x

    train_tok = train_ds.map(tokenize, batched=True, remove_columns=train_ds.column_names)
    dev_tok = dev_ds.map(tokenize, batched=True, remove_columns=dev_ds.column_names)

    # Training Args
    args = Seq2SeqTrainingArguments(
        output_dir=out_dir,
        learning_rate=lr,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        save_strategy="epoch",
        eval_strategy="epoch",
        logging_steps=50,
        report_to=[],
        fp16=torch.cuda.is_available(),
    )

    # Trainer
    collator = DataCollatorForSeq2Seq(tok, model=model)
    trainer = Seq2SeqTrainer(model=model, args=args, train_dataset=train_tok, eval_dataset=dev_tok, data_collator=collator)
    trainer.train()

    trainer.save_model(out_dir)
    tok.save_pretrained(out_dir)
    return out_dir

### Run stage 1 fine-tuning

In [17]:
IDIOM_ONLY_DIR = os.path.join(CHECKPOINT_DIR, "idiom_only_v1")

# Fine-tuning Hyperparameters
lr = 5e-5
epochs = 3
batch_size = 8

# Run Fine-tuning or reuse checkpoint
if checkpoint_exists(IDIOM_ONLY_DIR) and not FORCE_RETRAIN:
    print(f"Loading existing idiom-only checkpoint from: {IDIOM_ONLY_DIR}")
else:
    print(f"Training idiom-only model and saving to: {IDIOM_ONLY_DIR}")
    fine_tune(BASE_MODEL, idiom_ds["train"], idiom_ds["dev"], IDIOM_ONLY_DIR, lr=lr, epochs=epochs, batch_size=batch_size)

idiom_model = clear_generation_max_length(AutoModelForSeq2SeqLM.from_pretrained(IDIOM_ONLY_DIR).to(device))
idiom_tok = AutoTokenizer.from_pretrained(IDIOM_ONLY_DIR, use_fast=False)

Loading existing idiom-only checkpoint from: /content/drive/MyDrive/ds266_idiom_mt/checkpoints/idiom_only_v1


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

### Stage 1 Evaluation

In [18]:
# Evaluate Idiom Translation after stage 1 fine-tuning
idiom_pred = generate_preds(idiom_model, idiom_tok, idiom_test["src"],
                            max_source_len=MAX_SOURCE_LEN,
                            max_new_tokens=MAX_NEW_TOKENS,
                            batch_size=16)
idiom_m = compute_metrics(idiom_pred, idiom_test["tgt"])
print("idiom_only idioms:", idiom_m)

log_metrics(
    "idiom_only_v1",
    "idioms_test",
    idiom_m,
    model_name=IDIOM_ONLY_DIR,
    learning_rate=lr,
    epochs=epochs,
    batch_size=batch_size,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(idiom_ds["train"]),
    dev_size=len(idiom_ds["dev"]),
    seed=SEED,
    n_eval=len(idiom_test),
    notes="stage1 idiom-only fine-tune",
)

upsert_canonical_metric(
    "idiom_only_v1",
    "idioms_test",
    idiom_m,
    source_notebook="01_experiment_idiom_aware_MT_two_stage_frozen_encoder.ipynb",
    source_run_name="idiom_only_v1",
    notes="stage1 idiom-only fine-tune",
)

# Evaluate General Translation after stage 1 fine-tuning
wmt_pred = generate_preds(idiom_model, idiom_tok, wmt_test["src"],
                          max_source_len=MAX_SOURCE_LEN,
                          max_new_tokens=MAX_NEW_TOKENS,
                          batch_size=16)
wmt_m = compute_metrics(wmt_pred, wmt_test["tgt"])
print("idiom_only wmt:", wmt_m)

log_metrics(
    "idiom_only_v1",
    "wmt_test",
    wmt_m,
    model_name=IDIOM_ONLY_DIR,
    learning_rate=lr,
    epochs=epochs,
    batch_size=batch_size,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=False,
    train_size=len(general_ds["ft_train"]),
    dev_size=len(general_ds["ft_dev"]),
    seed=SEED,
    n_eval=len(wmt_test),
    notes="stage1 idiom-only fine-tune (eval on WMT)",
)

upsert_canonical_metric(
    "idiom_only_v1",
    "wmt_test",
    wmt_m,
    source_notebook="01_experiment_idiom_aware_MT_two_stage_frozen_encoder.ipynb",
    source_run_name="idiom_only_v1",
    notes="stage1 idiom-only fine-tune (eval on WMT)",
)

idiom_only idioms: {'bleu': 44.16121039160048, 'chrf': 64.43806053496827}
idiom_only wmt: {'bleu': 26.540146766159747, 'chrf': 58.23095853686714}


,timestamp,model,model_label,split,bleu,chrf,source_notebook,source_run_name,notes
0,2026-03-28T21:15:21-07:00,idiom_only_v1,idiom_only_v1,wmt_test,26.540147,58.230959,01_experiment_idiom_aware_MT_two_stage_frozen_...,idiom_only_v1,stage1 idiom-only fine-tune (eval on WMT)


## Stage 2: General Language fine-tuning
with Frozen Encoder option

In [19]:
STAGE1 = os.path.join(CHECKPOINT_DIR, "two_stage_frozen_v1_stage1")
STAGE2 = os.path.join(CHECKPOINT_DIR, "two_stage_frozen_v1_stage2")

# Stage 1 hyperparameters
lr_stage1 = 5e-5
epochs_stage1 = 3
batch_size_stage1 = 8

# Stage 2 hyperparameters
lr_stage2 = 1e-5
epochs_stage2 = 3
batch_size_stage2 = 8

# Stage 1: idiom-only fine-tuning
if checkpoint_exists(STAGE1) and not FORCE_RETRAIN:
    print(f"Loading existing stage-1 checkpoint from: {STAGE1}")
else:
    print(f"Training stage-1 checkpoint and saving to: {STAGE1}")
    fine_tune(
        BASE_MODEL,
        idiom_ds["train"],
        idiom_ds["dev"],
        STAGE1,
        lr=lr_stage1,
        epochs=epochs_stage1,
        batch_size=batch_size_stage1,
    )

# Stage 2: continue training on WMT with encoder frozen
if checkpoint_exists(STAGE2) and not FORCE_RETRAIN:
    print(f"Loading existing stage-2 checkpoint from: {STAGE2}")
else:
    print(f"Training stage-2 checkpoint and saving to: {STAGE2}")
    fine_tune(
        STAGE1,
        general_ds["ft_train"],
        general_ds["ft_dev"],
        STAGE2,
        lr=lr_stage2,
        epochs=epochs_stage2,
        batch_size=batch_size_stage2,
        freeze_encoder=True,
    )

f_model = clear_generation_max_length(
    AutoModelForSeq2SeqLM.from_pretrained(STAGE2).to(device)
)
f_tok = AutoTokenizer.from_pretrained(STAGE2, use_fast=False)
print("Stage-2 checkpoint ready:", STAGE2)


Loading existing stage-1 checkpoint from: /content/drive/MyDrive/ds266_idiom_mt/checkpoints/two_stage_frozen_v1_stage1
Loading existing stage-2 checkpoint from: /content/drive/MyDrive/ds266_idiom_mt/checkpoints/two_stage_frozen_v1_stage2


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Stage-2 checkpoint ready: /content/drive/MyDrive/ds266_idiom_mt/checkpoints/two_stage_frozen_v1_stage2


### Stage 2 Evaluation

In [20]:
# Evaluate Idiom Translation after stage 2 fine-tuning
idiom_pred = generate_preds(f_model, f_tok, idiom_test["src"],
                            max_source_len=MAX_SOURCE_LEN,
                            max_new_tokens=MAX_NEW_TOKENS,
                            batch_size=16)
idiom_m = compute_metrics(idiom_pred, idiom_test["tgt"])
print("two_stage_frozen idioms:", idiom_m)

log_metrics(
    "two_stage_frozen_v1",
    "idioms_test",
    idiom_m,
    model_name=STAGE2,
    learning_rate=lr_stage2,
    epochs=epochs_stage2,
    batch_size=batch_size_stage2,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=True,
    train_size=len(general_ds["ft_train"]),
    dev_size=len(general_ds["ft_dev"]),
    seed=SEED,
    n_eval=len(idiom_test),
    notes="stage2 general fine-tune, encoder frozen (eval on idioms)",
)

upsert_canonical_metric(
    "two_stage_frozen_v1",
    "idioms_test",
    idiom_m,
    source_notebook="01_experiment_idiom_aware_MT_two_stage_frozen_encoder.ipynb",
    source_run_name="two_stage_frozen_v1",
    notes="stage2 general fine-tune, encoder frozen (eval on idioms)",
)

# Evaluate General Translation after stage 2 fine-tuning
wmt_pred = generate_preds(f_model, f_tok, wmt_test["src"],
                          max_source_len=MAX_SOURCE_LEN,
                          max_new_tokens=MAX_NEW_TOKENS,
                          batch_size=16)
wmt_m = compute_metrics(wmt_pred, wmt_test["tgt"])
print("two_stage_frozen wmt:", wmt_m)

log_metrics(
    "two_stage_frozen_v1",
    "wmt_test",
    wmt_m,
    model_name=STAGE2,
    learning_rate=lr_stage2,
    epochs=epochs_stage2,
    batch_size=batch_size_stage2,
    max_source_len=MAX_SOURCE_LEN,
    max_new_tokens=MAX_NEW_TOKENS,
    freeze_encoder=True,
    train_size=len(general_ds["ft_train"]),
    dev_size=len(general_ds["ft_dev"]),
    seed=SEED,
    n_eval=len(wmt_test),
    notes="stage2 general fine-tune, encoder frozen (eval on WMT)",
)

upsert_canonical_metric(
    "two_stage_frozen_v1",
    "wmt_test",
    wmt_m,
    source_notebook="01_experiment_idiom_aware_MT_two_stage_frozen_encoder.ipynb",
    source_run_name="two_stage_frozen_v1",
    notes="stage2 general fine-tune, encoder frozen (eval on WMT)",
)

two_stage_frozen idioms: {'bleu': 43.15978726962502, 'chrf': 63.341617550181375}
two_stage_frozen wmt: {'bleu': 27.182380871327467, 'chrf': 58.34207281103036}


,timestamp,model,model_label,split,bleu,chrf,source_notebook,source_run_name,notes
0,2026-03-28T21:17:12-07:00,two_stage_frozen_v1,two_stage_frozen_v1,wmt_test,27.182381,58.342073,01_experiment_idiom_aware_MT_two_stage_frozen_...,two_stage_frozen_v1,"stage2 general fine-tune, encoder frozen (eval..."


### Canonical metrics quick check

In [21]:
# Canonical metrics quick check
if os.path.exists(CANONICAL_METRICS_PATH):
    canonical_df = pd.read_csv(CANONICAL_METRICS_PATH)
    display(
        canonical_df[
            canonical_df["model_label"].isin(["baseline", "idiom_only_v1", "two_stage_frozen_v1"])
        ].sort_values(["model_label", "split"])
    )
    print("Canonical file:", CANONICAL_METRICS_PATH)
else:
    print("Canonical file not found yet:", CANONICAL_METRICS_PATH)


,timestamp,model,model_label,split,bleu,chrf,source_notebook,source_run_name,notes
0,2026-03-28T21:11:44-07:00,baseline,baseline,idioms_test,39.649352,60.786112,01_experiment_idiom_aware_MT_two_stage_frozen_...,baseline,pretrained baseline
1,2026-03-28T21:13:27-07:00,baseline,baseline,wmt_test,27.567994,58.431846,01_experiment_idiom_aware_MT_two_stage_frozen_...,baseline,pretrained baseline (WMT test full 3003)
2,2026-03-28T21:13:36-07:00,idiom_only_v1,idiom_only_v1,idioms_test,44.161210,64.438061,01_experiment_idiom_aware_MT_two_stage_frozen_...,idiom_only_v1,stage1 idiom-only fine-tune
3,2026-03-28T21:15:21-07:00,idiom_only_v1,idiom_only_v1,wmt_test,26.540147,58.230959,01_experiment_idiom_aware_MT_two_stage_frozen_...,idiom_only_v1,stage1 idiom-only fine-tune (eval on WMT)
4,2026-03-28T21:15:29-07:00,two_stage_frozen_v1,two_stage_frozen_v1,idioms_test,43.159787,63.341618,01_experiment_idiom_aware_MT_two_stage_frozen_...,two_stage_frozen_v1,"stage2 general fine-tune, encoder frozen (eval..."
5,2026-03-28T21:17:12-07:00,two_stage_frozen_v1,two_stage_frozen_v1,wmt_test,27.182381,58.342073,01_experiment_idiom_aware_MT_two_stage_frozen_...,two_stage_frozen_v1,"stage2 general fine-tune, encoder frozen (eval..."


Canonical file: /content/drive/MyDrive/ds266_idiom_mt/results/automatic_metrics_canonical.csv


### Checkpoint status quick check

In [22]:
for label, path in {
    "IDIOM_ONLY_DIR": IDIOM_ONLY_DIR if 'IDIOM_ONLY_DIR' in globals() else None,
    "STAGE1": STAGE1 if 'STAGE1' in globals() else None,
    "STAGE2": STAGE2 if 'STAGE2' in globals() else None,
}.items():
    print(label, "->", path)
    if path is not None:
        print("exists:", checkpoint_exists(path))
    print("---")

IDIOM_ONLY_DIR -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/idiom_only_v1
exists: True
---
STAGE1 -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/two_stage_frozen_v1_stage1
exists: True
---
STAGE2 -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/two_stage_frozen_v1_stage2
exists: True
---


### Checkpoint status quick check

In [23]:
for label, path in {
    "IDIOM_ONLY_DIR": IDIOM_ONLY_DIR if 'IDIOM_ONLY_DIR' in globals() else None,
    "STAGE2": STAGE2 if 'STAGE2' in globals() else None,
}.items():
    print(label, "->", path)
    if path is not None:
        print("exists:", checkpoint_exists(path))
    print("---")

IDIOM_ONLY_DIR -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/idiom_only_v1
exists: True
---
STAGE2 -> /content/drive/MyDrive/ds266_idiom_mt/checkpoints/two_stage_frozen_v1_stage2
exists: True
---
